### **Imports, configuration and Silver data loading**

In [6]:
# ============================================================
# BANKING TRANSACTION AND FRAUD ANALYTICS
# GOLD LAYER TRANSFORMATION
# ============================================================

#IMPORTS

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window


#LAKEHOUSE CONFIGURATION

SILVER_LAKEHOUSE_PATH = (
    "abfss://Banking_Fraud_Analytics"
    "@onelake.dfs.fabric.microsoft.com/"
    "banking_silver_lh.Lakehouse"
)

SILVER_TABLES_PATH = (
    f"{SILVER_LAKEHOUSE_PATH}/Tables/dbo"
)


#SILVER SOURCE TABLES

SILVER_TABLES = {
    "account": "silver_account",
    "card": "silver_card",
    "client": "silver_client",
    "disp": "silver_disp",
    "district": "silver_district",
    "loan": "silver_loan",
    "orders": "silver_orders",
    "trans": "silver_trans",
    "paysim": "silver_paysim"
}


#LOAD SILVER DATASETS

silver_dfs = {}

for dataset_name, table_name in SILVER_TABLES.items():

    table_path = f"{SILVER_TABLES_PATH}/{table_name}"

    silver_dfs[dataset_name] = (
        spark.read
        .format("delta")
        .load(table_path)
    )

    print(f"Loaded: {table_name}")


#VERIFY SOURCE DATASETS

print("\nSILVER SOURCE DATA SUMMARY")
print("=" * 45)

for dataset_name, dataframe in silver_dfs.items():

    print(
        f"{dataset_name:<12} "
        f"{dataframe.count():>12,} records"
    )

StatementMeta(, 33cffe84-0ef3-437c-8ad9-853bdfe101af, 8, Finished, Available, Finished, False)

Loaded: silver_account
Loaded: silver_card
Loaded: silver_client
Loaded: silver_disp
Loaded: silver_district
Loaded: silver_loan
Loaded: silver_orders
Loaded: silver_trans
Loaded: silver_paysim

SILVER SOURCE DATA SUMMARY
account             4,500 records
card                  892 records
client              5,369 records
disp                5,369 records
district               77 records
loan                  682 records
orders              6,471 records
trans           1,056,320 records
paysim          6,362,620 records


### **Build the Gold dimensions**


In [7]:
# ============================================================
# BUILD GOLD DIMENSIONS
# ============================================================

district_window = Window.orderBy("district_id")

dim_district = (
    silver_dfs["district"]
    .dropDuplicates(["district_id"])
    .withColumn(
        "district_key",
        F.row_number().over(district_window)
    )
    .select(
        "district_key",
        "district_id",
        "district_name",
        "region",
        "population",
        "municipalities_under_500",
        "municipalities_500_to_1999",
        "municipalities_2000_to_9999",
        "municipalities_over_10000",
        "cities",
        "urban_population_pct",
        "average_salary",
        "unemployment_rate_1995",
        "unemployment_rate_1996",
        "entrepreneurs_per_1000",
        "crimes_1995",
        "crimes_1996"
    )
)


# 2. CUSTOMER DIMENSION

customer_window = Window.orderBy("client_id")

dim_customer = (
    silver_dfs["client"]
    .select("client_id", "district_id")
    .dropDuplicates(["client_id"])
    .withColumn(
        "customer_key",
        F.row_number().over(customer_window)
    )
    .join(
        dim_district.select("district_id", "district_key"),
        on="district_id",
        how="left"
    )
    .select(
        "customer_key",
        "client_id",
        "district_id",
        "district_key"
    )
)


# 3. ACCOUNT DIMENSION

account_window = Window.orderBy("account_id")

dim_account = (
    silver_dfs["account"]
    .select(
        "account_id",
        "district_id",
        F.col("frequency").alias("statement_frequency"),
        F.col("date").alias("account_open_date")
    )
    .dropDuplicates(["account_id"])
    .withColumn(
        "account_key",
        F.row_number().over(account_window)
    )
    .join(
        dim_district.select("district_id", "district_key"),
        on="district_id",
        how="left"
    )
    .select(
        "account_key",
        "account_id",
        "district_id",
        "district_key",
        "statement_frequency",
        "account_open_date"
    )
)


# 4. DATE DIMENSION

date_bounds = (
    silver_dfs["trans"]
    .select(
        F.min("date").alias("min_date"),
        F.max("date").alias("max_date")
    )
    .first()
)

date_sequence = spark.createDataFrame(
    [(date_bounds["min_date"], date_bounds["max_date"])],
    ["min_date", "max_date"]
)

dim_date = (
    date_sequence
    .select(
        F.explode(
            F.sequence(
                F.col("min_date"),
                F.col("max_date"),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("full_date")
    )
    .withColumn(
        "date_key",
        F.date_format("full_date", "yyyyMMdd").cast("int")
    )
    .withColumn("year", F.year("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("day_of_week", F.dayofweek("full_date"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .withColumn("week_of_year", F.weekofyear("full_date"))
    .withColumn(
        "is_weekend",
        F.dayofweek("full_date").isin([1, 7])
    )
    .select(
        "date_key",
        "full_date",
        "year",
        "quarter",
        "month",
        "month_name",
        "day",
        "day_of_week",
        "day_name",
        "week_of_year",
        "is_weekend"
    )
)


# ============================================================
# DIMENSION SUMMARY
# ============================================================

DIMENSIONS = {
    "dim_district": dim_district,
    "dim_customer": dim_customer,
    "dim_account": dim_account,
    "dim_date": dim_date
}

print("GOLD DIMENSION SUMMARY")
print("=" * 45)

for table_name, dataframe in DIMENSIONS.items():
    print(f"{table_name}: {dataframe.count():,} records")

StatementMeta(, 33cffe84-0ef3-437c-8ad9-853bdfe101af, 9, Finished, Available, Finished, False)

GOLD DIMENSION SUMMARY
dim_district: 77 records
dim_customer: 5,369 records
dim_account: 4,500 records
dim_date: 2,191 records


### **Build the customer-account bridge**

In [8]:
# ============================================================
# CUSTOMER-ACCOUNT BRIDGE
# ============================================================

# Prepare source relationships
customer_account_source = (
    silver_dfs["disp"]
    .select(
        F.col("client_id").cast("long"),
        F.col("account_id").cast("long"),
        F.col("type").alias("relationship_type")
    )
    .dropDuplicates([
        "client_id",
        "account_id",
        "relationship_type"
    ])
)

# Resolve customer and account surrogate keys
bridge_customer_account = (
    customer_account_source
    .join(
        dim_customer.select("client_id", "customer_key"),
        on="client_id",
        how="left"
    )
    .join(
        dim_account.select("account_id", "account_key"),
        on="account_id",
        how="left"
    )
    .select(
        "customer_key",
        "account_key",
        "relationship_type"
    )
)


# ============================================================
# BRIDGE VALIDATION
# ============================================================

print(
    "Bridge records:",
    bridge_customer_account.count()
)

print(
    "Missing customer keys:",
    bridge_customer_account
    .filter(F.col("customer_key").isNull())
    .count()
)

print(
    "Missing account keys:",
    bridge_customer_account
    .filter(F.col("account_key").isNull())
    .count()
)

print("\nRelationship distribution:")

bridge_customer_account.groupBy(
    "relationship_type"
).count().orderBy("relationship_type").show()

StatementMeta(, 33cffe84-0ef3-437c-8ad9-853bdfe101af, 10, Finished, Available, Finished, False)

Bridge records: 5369
Missing customer keys: 0
Missing account keys: 0

Relationship distribution:
+-----------------+-----+
|relationship_type|count|
+-----------------+-----+
|        DISPONENT|  869|
|            OWNER| 4500|
+-----------------+-----+



### **Build the Berka banking transaction fact**

In [9]:
# ============================================================
# BANK TRANSACTION FACT
# ============================================================

# 1. PREPARE SOURCE TRANSACTIONS

bank_transactions = (
    silver_dfs["trans"]
    .select(
        F.col("trans_id").cast("long"),
        F.col("account_id").cast("long"),
        F.col("date").alias("transaction_date"),
        F.col("type").alias("transaction_type"),
        "operation",
        "amount",
        "balance",
        "k_symbol",
        F.col("bank").alias("counterparty_bank"),
        F.col("account").alias("counterparty_account")
    )
)


# 2. RESOLVE DIMENSION KEYS

fact_bank_transactions = (
    bank_transactions
    .join(
        dim_account.select("account_id", "account_key"),
        on="account_id",
        how="left"
    )
    .join(
        dim_date.select(
            F.col("full_date").alias("transaction_date"),
            "date_key"
        ),
        on="transaction_date",
        how="left"
    )
    .select(
        "trans_id",
        "account_key",
        "date_key",
        "transaction_type",
        "operation",
        "amount",
        "balance",
        "k_symbol",
        "counterparty_bank",
        "counterparty_account"
    )
)


# ============================================================
# 3. FACT VALIDATION
# ============================================================

print("BANK TRANSACTION FACT VALIDATION")
print("=" * 45)

print(
    "Total transactions:",
    fact_bank_transactions.count()
)

print(
    "Unique transaction IDs:",
    fact_bank_transactions
    .select("trans_id")
    .distinct()
    .count()
)

print(
    "Duplicate transaction IDs:",
    fact_bank_transactions
    .groupBy("trans_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Missing account keys:",
    fact_bank_transactions
    .filter(F.col("account_key").isNull())
    .count()
)

print(
    "Missing date keys:",
    fact_bank_transactions
    .filter(F.col("date_key").isNull())
    .count()
)

StatementMeta(, 33cffe84-0ef3-437c-8ad9-853bdfe101af, 11, Finished, Available, Finished, False)

BANK TRANSACTION FACT VALIDATION
Total transactions: 1056320
Unique transaction IDs: 1056320
Duplicate transaction IDs: 0
Missing account keys: 0
Missing date keys: 0


### **Build the PaySim fraud transaction fact**

In [10]:
# ============================================================
# PAYSIM FRAUD TRANSACTION FACT
# ============================================================

# 1. DEFINE SOURCE COLUMNS FOR THE TRANSACTION HASH

PAYSIM_HASH_COLUMNS = [
    "step",
    "type",
    "amount",
    "nameOrig",
    "oldbalanceOrg",
    "newbalanceOrig",
    "nameDest",
    "oldbalanceDest",
    "newbalanceDest",
    "isFraud",
    "isFlaggedFraud"
]


# 2. GENERATE TRANSACTION IDENTIFIER

paysim_source = silver_dfs["paysim"]

paysim_with_id = paysim_source.withColumn(
    "fraud_transaction_id",
    F.sha2(
        F.concat_ws(
            "||",
            *[
                F.coalesce(
                    F.col(column_name).cast("string"),
                    F.lit("<NULL>")
                )
                for column_name in PAYSIM_HASH_COLUMNS
            ]
        ),
        256
    )
)


# 3. SELECT AND RENAME FACT COLUMNS

fact_fraud_transactions = (
    paysim_with_id
    .select(
        "fraud_transaction_id",
        F.col("step").alias("simulation_step"),
        F.col("type").alias("transaction_type"),
        "amount",
        F.col("nameOrig").alias("origin_account"),
        F.col("oldbalanceOrg").alias("origin_old_balance"),
        F.col("newbalanceOrig").alias("origin_new_balance"),
        F.col("nameDest").alias("destination_account"),
        F.col("oldbalanceDest").alias("destination_old_balance"),
        F.col("newbalanceDest").alias("destination_new_balance"),
        F.col("isFraud").alias("is_fraud"),
        F.col("isFlaggedFraud").alias("is_flagged_fraud")
    )
)


# ============================================================
# 4. FRAUD FACT VALIDATION
# ============================================================

print("PAYSIM FRAUD FACT VALIDATION")
print("=" * 45)

print(
    "Total transactions:",
    fact_fraud_transactions.count()
)

print(
    "Unique transaction IDs:",
    fact_fraud_transactions
    .select("fraud_transaction_id")
    .distinct()
    .count()
)

print(
    "Duplicate transaction IDs:",
    fact_fraud_transactions
    .groupBy("fraud_transaction_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Missing transaction IDs:",
    fact_fraud_transactions
    .filter(F.col("fraud_transaction_id").isNull())
    .count()
)

print("\nFraud distribution:")

fact_fraud_transactions.groupBy("is_fraud").count().orderBy("is_fraud").show()

print("Flagged fraud distribution:")

fact_fraud_transactions.groupBy("is_flagged_fraud").count().orderBy("is_flagged_fraud").show()

StatementMeta(, 33cffe84-0ef3-437c-8ad9-853bdfe101af, 12, Finished, Available, Finished, False)

PAYSIM FRAUD FACT VALIDATION
Total transactions: 6362620
Unique transaction IDs: 6362620
Duplicate transaction IDs: 0
Missing transaction IDs: 0

Fraud distribution:
+--------+-------+
|is_fraud|  count|
+--------+-------+
|       0|6354407|
|       1|   8213|
+--------+-------+

Flagged fraud distribution:
+----------------+-------+
|is_flagged_fraud|  count|
+----------------+-------+
|               0|6362604|
|               1|     16|
+----------------+-------+



### **GOLD MODEL VALIDATION**

In [11]:
# ============================================================
# GOLD MODEL VALIDATION
# ============================================================

def validate_dimension_key(
    dataframe: DataFrame,
    key_column: str,
    table_name: str
) -> None:
    """Check dimension key uniqueness and null values."""

    total_rows = dataframe.count()

    unique_keys = (
        dataframe
        .select(key_column)
        .distinct()
        .count()
    )

    null_keys = (
        dataframe
        .filter(F.col(key_column).isNull())
        .count()
    )

    duplicate_keys = total_rows - unique_keys

    print(f"\n{table_name}")
    print("-" * 40)
    print("Total rows:", total_rows)
    print("Unique keys:", unique_keys)
    print("Duplicate keys:", duplicate_keys)
    print("Null keys:", null_keys)

    if duplicate_keys > 0 or null_keys > 0:
        raise ValueError(f"Key validation failed for {table_name}")


def count_missing_references(
    source_df: DataFrame,
    target_df: DataFrame,
    source_key: str,
    target_key: str
) -> int:
    """Count source records without a matching target key."""

    target_keys = (
        target_df
        .select(F.col(target_key).alias("_target_key"))
        .distinct()
    )

    return (
        source_df
        .join(
            target_keys,
            F.col(source_key) == F.col("_target_key"),
            "left"
        )
        .filter(F.col("_target_key").isNull())
        .count()
    )


# ============================================================
# 1. DIMENSION PRIMARY KEYS
# ============================================================

print("DIMENSION KEY VALIDATION")
print("=" * 50)

validate_dimension_key(dim_district, "district_key", "dim_district")
validate_dimension_key(dim_customer, "customer_key", "dim_customer")
validate_dimension_key(dim_account, "account_key", "dim_account")
validate_dimension_key(dim_date, "date_key", "dim_date")


# ============================================================
# 2. DIMENSION FOREIGN KEYS
# ============================================================

print("\nDIMENSION RELATIONSHIP VALIDATION")
print("=" * 50)

customer_district_missing = count_missing_references(
    dim_customer,
    dim_district,
    "district_key",
    "district_key"
)

account_district_missing = count_missing_references(
    dim_account,
    dim_district,
    "district_key",
    "district_key"
)

print("Customer district references missing:", customer_district_missing)
print("Account district references missing:", account_district_missing)


# ============================================================
# 3. CUSTOMER-ACCOUNT BRIDGE
# ============================================================

print("\nBRIDGE TABLE VALIDATION")
print("=" * 50)

bridge_customer_missing = count_missing_references(
    bridge_customer_account,
    dim_customer,
    "customer_key",
    "customer_key"
)

bridge_account_missing = count_missing_references(
    bridge_customer_account,
    dim_account,
    "account_key",
    "account_key"
)

print("Missing customer references:", bridge_customer_missing)
print("Missing account references:", bridge_account_missing)


# ============================================================
# 4. BANK TRANSACTION FACT
# ============================================================

print("\nBANK TRANSACTION FACT VALIDATION")
print("=" * 50)

bank_account_missing = count_missing_references(
    fact_bank_transactions,
    dim_account,
    "account_key",
    "account_key"
)

bank_date_missing = count_missing_references(
    fact_bank_transactions,
    dim_date,
    "date_key",
    "date_key"
)

bank_duplicate_ids = (
    fact_bank_transactions.count()
    - fact_bank_transactions.select("trans_id").distinct().count()
)

print("Transaction records:", fact_bank_transactions.count())
print("Duplicate transaction IDs:", bank_duplicate_ids)
print("Missing account references:", bank_account_missing)
print("Missing date references:", bank_date_missing)


# ============================================================
# 5. PAYSIM FRAUD FACT
# ============================================================

print("\nFRAUD TRANSACTION FACT VALIDATION")
print("=" * 50)

fraud_total = fact_fraud_transactions.count()

fraud_unique_ids = (
    fact_fraud_transactions
    .select("fraud_transaction_id")
    .distinct()
    .count()
)

fraud_missing_ids = (
    fact_fraud_transactions
    .filter(F.col("fraud_transaction_id").isNull())
    .count()
)

fraud_duplicate_ids = fraud_total - fraud_unique_ids

print("Fraud transaction records:", fraud_total)
print("Duplicate transaction IDs:", fraud_duplicate_ids)
print("Missing transaction IDs:", fraud_missing_ids)


# ============================================================
# 6. FINAL VALIDATION RESULT
# ============================================================

validation_results = {
    "customer_district_missing": customer_district_missing,
    "account_district_missing": account_district_missing,
    "bridge_customer_missing": bridge_customer_missing,
    "bridge_account_missing": bridge_account_missing,
    "bank_account_missing": bank_account_missing,
    "bank_date_missing": bank_date_missing,
    "bank_duplicate_ids": bank_duplicate_ids,
    "fraud_duplicate_ids": fraud_duplicate_ids,
    "fraud_missing_ids": fraud_missing_ids
}

failed_checks = {
    check_name: issue_count
    for check_name, issue_count in validation_results.items()
    if issue_count > 0
}

print("\nFINAL GOLD MODEL VALIDATION")
print("=" * 50)

if failed_checks:
    print("VALIDATION FAILED")
    for check_name, issue_count in failed_checks.items():
        print(f"{check_name}: {issue_count}")
    raise ValueError("Resolve Gold model validation issues before saving.")

print("All Gold model validation checks passed.")

StatementMeta(, 33cffe84-0ef3-437c-8ad9-853bdfe101af, 13, Finished, Available, Finished, False)

DIMENSION KEY VALIDATION

dim_district
----------------------------------------
Total rows: 77
Unique keys: 77
Duplicate keys: 0
Null keys: 0

dim_customer
----------------------------------------
Total rows: 5369
Unique keys: 5369
Duplicate keys: 0
Null keys: 0

dim_account
----------------------------------------
Total rows: 4500
Unique keys: 4500
Duplicate keys: 0
Null keys: 0

dim_date
----------------------------------------
Total rows: 2191
Unique keys: 2191
Duplicate keys: 0
Null keys: 0

DIMENSION RELATIONSHIP VALIDATION
Customer district references missing: 0
Account district references missing: 0

BRIDGE TABLE VALIDATION
Missing customer references: 0
Missing account references: 0

BANK TRANSACTION FACT VALIDATION
Transaction records: 1056320
Duplicate transaction IDs: 0
Missing account references: 0
Missing date references: 0

FRAUD TRANSACTION FACT VALIDATION
Fraud transaction records: 6362620
Duplicate transaction IDs: 0
Missing transaction IDs: 0

FINAL GOLD MODEL VALIDAT

### **Save all Gold tables**

In [12]:
# ============================================================
# SAVE GOLD TABLES
# ============================================================

GOLD_TABLES = {
    "dim_district": dim_district,
    "dim_customer": dim_customer,
    "dim_account": dim_account,
    "dim_date": dim_date,
    "bridge_customer_account": bridge_customer_account,
    "fact_bank_transactions": fact_bank_transactions,
    "fact_fraud_transactions": fact_fraud_transactions
}


# ============================================================
# WRITE TABLES TO GOLD LAKEHOUSE
# ============================================================

print("SAVING GOLD TABLES")
print("=" * 50)

for table_name, dataframe in GOLD_TABLES.items():

    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    print(f"Saved successfully: {table_name}")

print("\nAll Gold tables saved.")



StatementMeta(, 33cffe84-0ef3-437c-8ad9-853bdfe101af, 14, Finished, Available, Finished, False)

SAVING GOLD TABLES
Saved successfully: dim_district
Saved successfully: dim_customer
Saved successfully: dim_account
Saved successfully: dim_date
Saved successfully: bridge_customer_account
Saved successfully: fact_bank_transactions
Saved successfully: fact_fraud_transactions

All Gold tables saved.


### **Verify the saved Gold tables**

In [13]:
# ============================================================
# VERIFY SAVED GOLD TABLES
# ============================================================

EXPECTED_COUNTS = {
    "dim_district": 77,
    "dim_customer": 5369,
    "dim_account": 4500,
    "dim_date": 2191,
    "bridge_customer_account": 5369,
    "fact_bank_transactions": 1056320,
    "fact_fraud_transactions": 6362620
}


# ============================================================
# CHECK SAVED TABLES
# ============================================================

print("GOLD LAKEHOUSE VERIFICATION")
print("=" * 60)

verification_results = []

for table_name, expected_count in EXPECTED_COUNTS.items():

    saved_df = spark.table(table_name)
    actual_count = saved_df.count()

    status = (
        "PASS"
        if actual_count == expected_count
        else "FAIL"
    )

    verification_results.append({
        "table_name": table_name,
        "expected_count": expected_count,
        "actual_count": actual_count,
        "status": status
    })

    print(
        f"{table_name:<30} "
        f"Expected: {expected_count:>10,} "
        f"Actual: {actual_count:>10,} "
        f"{status}"
    )


# ============================================================
# FINAL RESULT
# ============================================================

failed_tables = [
    result
    for result in verification_results
    if result["status"] == "FAIL"
]

print("\n" + "=" * 60)

if failed_tables:
    raise ValueError("Gold Lakehouse verification failed.")

print("All seven Gold tables verified successfully.")

StatementMeta(, 33cffe84-0ef3-437c-8ad9-853bdfe101af, 15, Finished, Available, Finished, False)

GOLD LAKEHOUSE VERIFICATION
dim_district                   Expected:         77 Actual:         77 PASS
dim_customer                   Expected:      5,369 Actual:      5,369 PASS
dim_account                    Expected:      4,500 Actual:      4,500 PASS
dim_date                       Expected:      2,191 Actual:      2,191 PASS
bridge_customer_account        Expected:      5,369 Actual:      5,369 PASS
fact_bank_transactions         Expected:  1,056,320 Actual:  1,056,320 PASS
fact_fraud_transactions        Expected:  6,362,620 Actual:  6,362,620 PASS

All seven Gold tables verified successfully.
